In [ ]:
# 2️⃣ Đọc dữ liệu Bronze
bronze_path = "dev.bronze_stream.bronze_streaming_item_properties"
df_bronze = spark.read.format("delta").load(bronze_path)
display(df_bronze.limit(5))  # Xem 5 bản ghi đầu tiên

In [ ]:
def get_spark():
    """Khởi tạo Spark session (dùng trên Databricks)."""
    return SparkSession.builder.appName("EventsSilver").getOrCreate()

# Khởi tạo Spark
spark = get_spark()


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Invalid Spark URL: spark://HeartbeatReceiver@BUI_DUY:60620
	at org.apache.spark.rpc.RpcEndpointAddress$.apply(RpcEndpointAddress.scala:66)
	at org.apache.spark.rpc.netty.NettyRpcEnv.asyncSetupEndpointRefByURI(NettyRpcEnv.scala:143)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.executor.Executor.<init>(Executor.scala:313)
	at org.apache.spark.scheduler.local.LocalEndpoint.<init>(LocalSchedulerBackend.scala:66)
	at org.apache.spark.scheduler.local.LocalSchedulerBackend.start(LocalSchedulerBackend.scala:136)
	at org.apache.spark.scheduler.TaskSchedulerImpl.start(TaskSchedulerImpl.scala:237)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:622)
	at org.apache.

In [ ]:
# 3️⃣ Kiểm tra thống kê các cột
print("Tổng số bản ghi:", df_bronze.count())
print("Số visitor duy nhất:", df_bronze.select("visitorid").distinct().count())
print("Số item duy nhất:", df_bronze.select("itemid").distinct().count())
df_bronze.groupBy("event").count().show()


In [ ]:
# 4️⃣ Silver Layer – Chuẩn hóa dữ liệu
df_silver = df_bronze.withColumn("event_time", from_unixtime(col("timestamp") / 1000)) \
    .withColumn("transactionid", when(col("event") == "transaction", col("transactionid")).otherwise(None)) \
    .filter(col("event").isin(["view", "addtocart", "transaction"]))

display(df_silver.limit(5))

In [ ]:
# 5️⃣ Trực quan hóa sự kiện theo thời gian
df_event_time = df_silver.groupBy("event_time").count().orderBy("event_time")
display(df_event_time)

In [ ]:
# 6️⃣ Phân tích visitor và item
df_visitor_count = df_silver.groupBy("visitorid").count().orderBy(col("count").desc())
display(df_visitor_count.limit(20))


In [ ]:
df_item_count = df_silver.groupBy("itemid").count().orderBy(col("count").desc())
display(df_item_count.limit(20))